In [1]:
import os
from dotenv import load_dotenv
import anthropic
import json

load_dotenv()
client = anthropic.Anthropic()
print("Setup complete")

Setup complete


In [2]:
extract_deal_tool = {
    "name": "extract_deal_info",
    "description": "Extract structured information about a commercial real estate deal from a description.",
    "input_schema": {
        "type": "object",
        "properties": {
            "deal_name": {
                "type": "string",
                "description": "The name or identifier of the deal."
            },
            "asset_type": {
                "type": "string",
                "enum": ["multifamily", "office", "retail", "industrial", "hotel", "mixed-use", "other"],
                "description": "The primary asset type of the deal."
            },
            "unit_count": {
                "type": "integer",
                "description": "Number of units (for multifamily/hotel) or null if not applicable."
            },
            "total_cost_usd": {
                "type": "number",
                "description": "Total project cost in US dollars. Convert millions/billions to actual numbers (e.g., $80M = 80000000)."
            },
            "location": {
                "type": "string",
                "description": "The city and state or region of the deal."
            },
            "risk_factors": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of key risk factors mentioned or implied in the description."
            }
        },
        "required": ["deal_name", "asset_type", "total_cost_usd", "location", "risk_factors"]
    }
}

print("Tool schema defined")

Tool schema defined


In [3]:
description = """
Project Oakwood is a new 200-unit luxury apartment development we're underwriting 
in Austin, TX. Total project cost is $80M. Key concerns include the heavy multifamily 
supply pipeline in the submarket, and our floating-rate construction loan exposure 
if rates rise further.
"""

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=[extract_deal_tool],
    tool_choice={"type": "tool", "name": "extract_deal_info"},
    messages=[
        {"role": "user", "content": f"Extract the deal information from this description:\n\n{description}"}
    ]
)

print("Stop reason:", response.stop_reason)
print()
for block in response.content:
    print(block)

Stop reason: tool_use

ToolUseBlock(id='toolu_01LPYv7BuQVrj9MciisdasAW', caller=DirectCaller(type='direct'), input={'deal_name': 'Project Oakwood', 'asset_type': 'multifamily', 'total_cost_usd': 80000000, 'location': 'Austin, TX', 'risk_factors': ['Heavy multifamily supply pipeline in the submarket', 'Floating-rate construction loan exposure if rates rise further'], 'unit_count': 200}, name='extract_deal_info', type='tool_use')


In [4]:
# Find the tool use block and extract the structured data
tool_use = next(block for block in response.content if block.type == "tool_use")
deal_data = tool_use.input

# Pretty-print it
print(json.dumps(deal_data, indent=2))

{
  "deal_name": "Project Oakwood",
  "asset_type": "multifamily",
  "total_cost_usd": 80000000,
  "location": "Austin, TX",
  "risk_factors": [
    "Heavy multifamily supply pipeline in the submarket",
    "Floating-rate construction loan exposure if rates rise further"
  ],
  "unit_count": 200
}


In [5]:
description2 = """
We got the OM on that Nashville office deal — 1515 Music Row.
It's a 180,000 SF class-A repositioning play at a total cost of just under $45 million.
Two risks I'm watching: the Nashville office market vacancy spike, 
and the fact that our anchor tenant's lease expires in Year 3.
"""

response2 = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=[extract_deal_tool],
    tool_choice={"type": "tool", "name": "extract_deal_info"},
    messages=[
        {"role": "user", "content": f"Extract the deal information:\n\n{description2}"}
    ]
)

deal_data2 = next(b for b in response2.content if b.type == "tool_use").input
print(json.dumps(deal_data2, indent=2))

{
  "deal_name": "1515 Music Row",
  "asset_type": "office",
  "total_cost_usd": 45000000,
  "location": "Nashville, TN",
  "risk_factors": [
    "Nashville office market vacancy spike",
    "Anchor tenant's lease expires in Year 3"
  ],
  "unit_count": "null"
}


In [6]:
description2 = """
We got the OM on that Nashville office deal — It's a 180,000 SF class-A repositioning play, Two risks I'm watching: the Nashville office market vacancy spike, 
and the fact that our anchor tenant's lease expires in Year 3, address is 1515 Music Row. Cost is a scotch under $45 million.
"""

response2 = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=[extract_deal_tool],
    tool_choice={"type": "tool", "name": "extract_deal_info"},
    messages=[
        {"role": "user", "content": f"Extract the deal information:\n\n{description2}"}
    ]
)

deal_data2 = next(b for b in response2.content if b.type == "tool_use").input
print(json.dumps(deal_data2, indent=2))

{
  "deal_name": "1515 Music Row",
  "asset_type": "office",
  "total_cost_usd": 45000000,
  "location": "Nashville, TN",
  "risk_factors": [
    "Nashville office market vacancy spike",
    "Anchor tenant lease expires in Year 3"
  ]
}


In [7]:
description = """
We're underwriting a mixed-use development at the corner of 3rd and Main in Charleston, SC.
Call it The Battery. 125 residential units above 40,000 SF of ground-floor retail.
All-in cost: roughly $62 million. Main risks are the retail lease-up in a tertiary market
and hurricane exposure on the coastal site.
"""

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=[extract_deal_tool],
    tool_choice={"type": "tool", "name": "extract_deal_info"},
    messages=[
        {"role": "user", "content": f"Extract the deal information:\n\n{description}"}
    ]
)

deal = next(b for b in response.content if b.type == "tool_use").input
print(json.dumps(deal, indent=2))

{
  "deal_name": "The Battery",
  "asset_type": "mixed-use",
  "total_cost_usd": 62000000,
  "location": "Charleston, SC",
  "risk_factors": [
    "Retail lease-up in a tertiary market",
    "Hurricane exposure on coastal site"
  ],
  "unit_count": 125
}


In [8]:
def extract_deal(description):
    """Extract structured deal info from an unstructured description."""
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=[extract_deal_tool],
        tool_choice={"type": "tool", "name": "extract_deal_info"},
        messages=[
            {"role": "user", "content": f"Extract the deal information:\n\n{description}"}
        ]
    )
    tool_use = next(b for b in response.content if b.type == "tool_use")
    return tool_use.input

# Test it on a single description
result = extract_deal("""
Houston retail center, 75,000 SF, $28M total. Anchor is Kroger on a
15-year NNN lease, but 40% of GLA comes up for renewal in the next 3 years.
""")

print(json.dumps(result, indent=2))

{
  "deal_name": "Houston retail center",
  "asset_type": "retail",
  "total_cost_usd": 28000000,
  "location": "Houston, TX",
  "risk_factors": [
    "40% of GLA (gross leasable area) comes up for renewal in next 3 years",
    "Significant near-term lease rollover risk",
    "Revenue concentration dependent on lease renewals"
  ],
  "unit_count": "null"
}


In [9]:
deals_in_pipeline = [
    "Project Summit — 340-unit multifamily development in Denver, $115M all-in cost. Risks: construction cost overruns and submarket concentration.",
    "The Axis — 220,000 SF suburban office in Raleigh at $85M, repositioning play. Major risk is the softening office market.",
    "Gateway Industrial — 450,000 SF last-mile logistics facility in Dallas, $52M. Primary risk is tenant credit — currently speculative.",
    "Harbor View — 180-key select-service hotel in San Diego, $67M conversion. Risks include lingering softness in business travel and FF&E contingency.",
]

# Extract all four deals in a loop
extracted = []
for i, deal_description in enumerate(deals_in_pipeline, 1):
    print(f"Extracting deal {i}...")
    extracted.append(extract_deal(deal_description))

# Print the results as a clean table of data
print()
for deal in extracted:
    print(f"  {deal['deal_name']:25} | {deal['asset_type']:12} | ${deal['total_cost_usd']:>12,} | {deal['location']}")

Extracting deal 1...
Extracting deal 2...
Extracting deal 3...
Extracting deal 4...

  Project Summit            | multifamily  | $ 115,000,000 | Denver
  The Axis                  | office       | $  85,000,000 | Raleigh
  Gateway Industrial        | industrial   | $  52,000,000 | Dallas, TX
  Harbor View               | hotel        | $  67,000,000 | San Diego


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
print(f"Key loaded: {api_key[:10]}..." if api_key else "Key NOT loaded")

Key loaded: sk-ant-api...
